# Nano-Jev v0.1 — mixed

Проверяем две предложенные схемы подачи примеров: смешивание с первого шага и чистый старт.

Одинаковые исходные веса, 22 656 примеров (5 664 clean + 16 992 simulated STT), E5 top-20, кандидаты, seed, один проход и 1 416 обновлений весов. В clean-first первые 142 обновления используют чистые примеры; после них перемешивается оставшийся состав. В обоих запусках каждая строка встречается один раз.

Валидация: 50 синтетических примеров каждого из basic, difficult и ASR. Профили оцениваются отдельно, test и личные каталоги остаются локально. Промахи E5 остаются ошибками всей цепочки.

После обучения проверяем checkpoint локально; GPU-время не подтверждает CPU-лимит 3 секунды.

## Среда

Используем GPU Kaggle и заранее закреплённые веса. Интернет нужен только для Python-зависимостей.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'transformers==5.6.2', 'safetensors==0.6.2'])
import torch
assert torch.cuda.is_available(), 'Нужен GPU Kaggle'
print({'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0)})


## Дообучение

Веса encoder и скалярной головы обновляются по cross-entropy выбора. Для обучения правильный объект может быть добавлен в shortlist; на валидации shortlist не исправляется. Baseline оценивается перед первым обновлением весов.

In [ ]:
matches = list(Path('/kaggle/input').rglob('experiment-input-manifest.json'))
assert len(matches) == 1, f'Ожидался один пакет данных, найдено {len(matches)}'
input_dir = matches[0].parent
output_dir = Path('/kaggle/working/results')
command = [sys.executable, str(input_dir / 'train_nanojev_kaggle.py'), '--input', str(input_dir), '--output', str(output_dir), '--schedule', 'mixed']
with Path('/kaggle/working/training.log').open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    assert process.wait() == 0, 'Обучение завершилось ошибкой; см. training.log'


## Результаты

Сравниваем baseline и checkpoint по каждому профилю. Один seed — первоначальный эксперимент; небольшая разница требует повторного подтверждения.

In [ ]:
result = json.loads((output_dir / 'result.json').read_text())
for profile in result['baseline']:
    before = result['baseline'][profile]['accuracy']
    after = result['trained'][profile]['accuracy']
    print(f'{profile}: {before:.1%} -> {after:.1%}; delta={(after-before)*100:+.1f} pp')
print({'steps': result['steps'], 'elapsed_seconds': result['elapsed_seconds'], 'checkpoint': str(output_dir / 'checkpoint')})
